In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_csv = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
      )

In [0]:
df_json = (
    spark
    .read
    .format("JSON")
    .option("inferSchema",True)
    .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
)

In [0]:
df_parquet = (
    spark
    .read
    .format("parquet")
    .load("/Volumes/learnspark/raw/spark_volume/raw_orders/part-00000-tid-orders.c000.snappy.parquet")
)

In [0]:
df_permissive = (
    spark
    .read
    .format("JSON")
    .option("inferSchema",True)
    .option("mode","PERMISSIVE")
    .load("/Volumes/gizmobox/landing/operational_data/orders/orders_2024_10.json")

)
display(df_permissive)

In [0]:
df_malformed = (
    spark
    .read
    .format("JSON")
    .option("inferSchema",True)
    .option("mode","DROPMALFORMED")
    .load("/Volumes/gizmobox/landing/operational_data/orders/orders_2024_10.json")

)
display(df_malformed)

In [0]:
df_fastfail = (
    spark
    .read
    .format("JSON")
    .option("inferSchema",True)
    .option("mode","FASTFAIL")
    .load("/Volumes/gizmobox/landing/operational_data/orders/orders_2024_10.json")

)
display(df_fastfail)

In [0]:
df_csv.schema

In [0]:
my_schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

In [0]:
df = (
    spark
    .read
    .format("csv")
    .schema(my_schema)
    .option("header",True)
    .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
)
display(df)

In [0]:
my_ddl_schema = """
order_id INT,
customer_id STRING,
order_date DATE,
product_id STRING,
quantity INTEGER,
price DOUBLE,
order_status STRING,
shipping_address STRING,
city STRING,
country STRING,
payment_method STRING,
discount DOUBLE,
category STRING,
sales_rep STRING,
region STRING,
ship_date DATE,
delivery_days INTEGER,
returned STRING,
gender STRING
"""
df_csv = (
    spark
    .read
    .format("csv")
    .schema(my_ddl_schema)
    .option("header",True)
    .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
)
display(df_csv)

In [0]:
df_selected = df_csv.select("order_id","order_date","order_status",col("product_id").alias("product"),"city","country","shipping_address",col("returned"),"price","quantity")

In [0]:
desired_orders = ["shipped","pending"]
df_filtered = (
    df_selected
    .filter(
        lower(col("order_status")).isin(desired_orders)
    )
)

df_column = (
    df_filtered.withColumnRenamed("order_status","status")
    .withColumn("flag", when(col("returned") == 'Yes',1).otherwise(0))
    .withColumn("shipping_address",regexp_replace(col("shipping_address"),",.*","")).alias("address")
    .withColumn("Total_price",round(col("price")*col("quantity"),3))
    .withColumn("city_number",regexp_replace(col("city"),"City","").cast("STRING"))
    .sort(["Total_price","city"])
    )
display(df_column)